# MangoLeaf AI: MobileNetV3-Large Transfer Learning Notebook
This notebook contains the complete machine learning pipeline to:
1. Mount Google Drive and load your combined leaf & fruit mango datasets.
2. Configure data generators with data augmentation on the training set (leaving test/validation sets clean).
3. Set up the pre-trained MobileNetV3-Large base model from Keras applications.
4. Fine-tune the classification head to recognize your 6 target classes (5 diseases + 1 healthy).
5. Save the trained model as `mangoleaf_model.h5`.
6. Convert the Keras model to compressed TensorFlow.js layer formats using Float16 weight quantization.

### Step 1: Mount Google Drive
Run this cell to grant Google Colab access to your Drive files where your dataset folders are stored.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Install TensorFlow.js Converter
This installs the command-line interface (CLI) tool needed to compile and shard the Keras model for browser loading.

In [ ]:
!pip install tensorflowjs

### Step 3: Imports & Directory Configuration
Make sure your dataset paths in Google Drive match this directory structure.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras import layers, models
import os

# Set path to your dataset folders in Google Drive
dataset_dir = '/content/drive/MyDrive/mango_dataset'
train_dir = os.path.join(dataset_dir, 'train')
val_dir = os.path.join(dataset_dir, 'validation')

print("Training path:", train_dir)
print("Validation path:", val_dir)

### Step 4: Data Augmentation & Generators
Here we apply random rotation, zooming, shifts, and flips to the training dataset. This increases the effective dataset size and helps prevent overfitting.

In [ ]:
# Data Augmentation on Training data
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Validation images are only rescaled (NO augmentation!)
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

### Step 5: Build MobileNetV3-Large Transfer Learning Model
We load the base model with pre-trained weights from ImageNet and freeze it. Then we attach our custom dense layers to classify the 6 disease categories.

In [ ]:
# Load pre-trained MobileNetV3-Large without the final classification head
base_model = MobileNetV3Large(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet',
    pooling='avg'
)

# Freeze the base model weights
base_model.trainable = False

# Add classification head for 6 target categories
model = models.Sequential([
    layers.Input(shape=(224, 224, 3)), # Fix for TF.js inputShape compatibility
    base_model,
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(6, activation='softmax')  # Softmax output layer
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

### Step 6: Train the Model
Run this cell to start training. The free T4 GPU will compile this very quickly.

In [ ]:
epochs = 15
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=val_generator
)

### Step 7: Save and Convert to TensorFlow.js
This saves the final Keras model and converts it using float16 quantization. The output is a folder containing a sharded set of files optimized for browser loading.

In [ ]:
# Save model as a standard TensorFlow SavedModel directory
model.save('mangoleaf_saved_model')
print('Saved Keras model to mangoleaf_saved_model folder')

# Convert to TensorFlow.js Graph format with Float16 compression
!tensorflowjs_converter \
    --input_format=tf_saved_model \
    --quantize_float16=* \
    mangoleaf_saved_model \
    ./tfjs_model
print('TensorFlow.js Graph model generated successfully inside ./tfjs_model')
